In [1]:
# Import dependencies
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset


# Define the residual block
class ResidualBlock(nn.Module):
    # Define initialization
    def __init__(self, hidden_dim, dropout):
        # Call parent constructor
        super().__init__()

        # Define first layer
        self.linear1 = nn.Linear(
            hidden_dim,
            hidden_dim,
        )

        # Define second layer
        self.linear2 = nn.Linear(
            hidden_dim,
            hidden_dim,
        )

        # Define activation
        self.activation = nn.ReLU()

        # Define dropout
        self.dropout = nn.Dropout(dropout)

        # Define normalization
        self.batch_norm = nn.BatchNorm1d(
            hidden_dim
        )

    # Define forward pass
    def forward(self, x):
        # Save residual
        residual = x

        # Apply first layer
        out = self.linear1(x)

        # Apply activation
        out = self.activation(out)

        # Apply dropout
        out = self.dropout(out)

        # Apply second layer
        out = self.linear2(out)

        # Add residual
        out = out + residual

        # Apply normalization
        out = self.batch_norm(out)

        # Apply activation
        out = self.activation(out)

        # Return output
        return out


# Define the residual network
class ResidualRegressor(nn.Module):
    # Define initialization
    def __init__(
        self,
        input_dim,
        hidden_dim,
        dropout,
    ):
        # Call parent constructor
        super().__init__()

        # Define input layer
        self.input_layer = nn.Linear(
            input_dim,
            hidden_dim,
        )

        # Define activation
        self.activation = nn.ReLU()

        # Define dropout
        self.dropout = nn.Dropout(dropout)

        # Define residual blocks
        self.block1 = ResidualBlock(
            hidden_dim,
            dropout,
        )

        self.block2 = ResidualBlock(
            hidden_dim,
            dropout,
        )

        self.block3 = ResidualBlock(
            hidden_dim,
            dropout,
        )

        # Define output layer
        self.output_layer = nn.Linear(
            hidden_dim,
            1,
        )

    # Define forward pass
    def forward(self, x):
        # Apply input layer
        x = self.input_layer(x)

        # Apply activation
        x = self.activation(x)

        # Apply dropout
        x = self.dropout(x)

        # Apply residual blocks
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)

        # Apply output layer
        x = self.output_layer(x)

        # Return output
        return x


# Define preprocessing function
def preprocess_data(
    train_df,
    test_df,
    target_col,
):
    # Split features
    X = train_df.drop(
        columns=[target_col]
    )

    # Extract target
    y = train_df[target_col]

    # Save test ids
    test_ids = test_df["id"]

    # Remove id column
    X = X.drop(columns=["id"])

    # Remove id from test
    X_test = test_df.drop(
        columns=["id"]
    )

    # Detect categorical columns
    categorical_cols = (
        X.select_dtypes(
            include=[
                "object",
                "category",
            ]
        )
        .columns.tolist()
    )

    # Detect numerical columns
    numerical_cols = (
        X.select_dtypes(
            exclude=[
                "object",
                "category",
            ]
        )
        .columns.tolist()
    )

    # Define numerical transformer
    numerical_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                ),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    # Define categorical transformer
    categorical_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                ),
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    # Define preprocessor
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numerical_transformer,
                numerical_cols,
            ),
            (
                "cat",
                categorical_transformer,
                categorical_cols,
            ),
        ]
    )

    # Fit and transform train
    X_processed = (
        preprocessor.fit_transform(X)
    )

    # Transform test
    X_test_processed = (
        preprocessor.transform(X_test)
    )

    # Return processed data
    return (
        X_processed,
        y.values,
        X_test_processed,
        test_ids,
    )


# Define training function
def train_model(
    model,
    train_loader,
    valid_loader,
    device,
    epochs,
    learning_rate,
):
    # Define loss function
    criterion = nn.L1Loss()

    # Define optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-5,
    )

    # Initialize best score
    best_mae = float("inf")

    # Initialize best weights
    best_state = None

    # Training loop
    for epoch in range(epochs):
        # Set train mode
        model.train()

        # Initialize train loss
        train_loss = 0.0

        # Iterate batches
        for batch_x, batch_y in train_loader:
            # Move to device
            batch_x = batch_x.to(device)

            # Move targets
            batch_y = batch_y.to(device)

            # Zero gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(
                batch_x
            ).squeeze()

            # Compute loss
            loss = criterion(
                outputs,
                batch_y,
            )

            # Backward pass
            loss.backward()

            # Optimizer step
            optimizer.step()

            # Accumulate loss
            train_loss += loss.item()

        # Set eval mode
        model.eval()

        # Initialize predictions
        predictions = []

        # Initialize targets
        targets = []

        # Disable gradients
        with torch.no_grad():
            # Iterate validation batches
            for batch_x, batch_y in valid_loader:
                # Move to device
                batch_x = batch_x.to(device)

                # Predict
                outputs = model(
                    batch_x
                ).squeeze()

                # Save predictions
                predictions.extend(
                    outputs.cpu().numpy()
                )

                # Save targets
                targets.extend(
                    batch_y.numpy()
                )

        # Compute MAE
        valid_mae = mean_absolute_error(
            targets,
            predictions,
        )

        # Print metrics
        print(
            f"Epoch {epoch + 1:02d} | "
            f"Train Loss: "
            f"{train_loss / len(train_loader):.5f} | "
            f"Valid MAE: "
            f"{valid_mae:.5f}"
        )

        # Save best model
        if valid_mae < best_mae:
            # Update best score
            best_mae = valid_mae

            # Save weights
            best_state = (
                model.state_dict()
            )

    # Load best weights
    model.load_state_dict(
        best_state
    )

    # Return model
    return model


# Define prediction function
def predict_model(
    model,
    X_test,
    device,
    batch_size=4096,
):
    # Convert tensor
    X_tensor = torch.tensor(
        X_test,
        dtype=torch.float32,
    )

    # Create loader
    loader = DataLoader(
        X_tensor,
        batch_size=batch_size,
        shuffle=False,
    )

    # Set eval mode
    model.eval()

    # Initialize predictions
    predictions = []

    # Disable gradients
    with torch.no_grad():
        # Iterate batches
        for batch_x in loader:
            # Move to device
            batch_x = batch_x.to(device)

            # Predict
            outputs = model(
                batch_x
            ).squeeze()

            # Save predictions
            predictions.extend(
                outputs.cpu().numpy()
            )

    # Return predictions
    return np.array(predictions)


# Define blending function
def blend_predictions(
    test_ids,
    prediction_dict,
    output_path,
    target_col,
):
    # Initialize weighted sum
    weighted_sum = np.zeros(
        len(test_ids)
    )

    # Initialize total weight
    total_weight = 0.0

    # Iterate predictions
    for name, config in (
        prediction_dict.items()
    ):
        # Extract predictions
        predictions = config[
            "predictions"
        ]

        # Extract weight
        weight = config[
            "weight"
        ]

        # Add weighted predictions
        weighted_sum += (
            predictions * weight
        )

        # Accumulate weight
        total_weight += weight

        # Print info
        print(
            f"Blending {name} | "
            f"Weight: {weight}"
        )

    # Compute final predictions
    final_predictions = (
        weighted_sum / total_weight
    )

    # Create submission
    submission = pd.DataFrame(
        {
            "id": test_ids,
            target_col: final_predictions,
        }
    )

    # Save submission
    submission.to_csv(
        output_path,
        index=False,
    )

    # Print confirmation
    print(
        f"✅ Submission saved to "
        f"{output_path}"
    )


# Define the main function
def main():
    # Define target
    TARGET = "PitNextLap"

    # Define competition path
    COMP_PATH = Path(
        "/kaggle/input/"
        "competitions/"
        "playground-series-s6e5"
    )

    # Define blend dataset path
    BLEND_PATH = Path(
        "/kaggle/input/"
        "datasets/"
        "anthonytherrien/"
        "predicting-f1-pit-stops-vault"
    )

    # Load train dataset
    train_df = pd.read_csv(
        COMP_PATH / "train.csv"
    )

    # Load test dataset
    test_df = pd.read_csv(
        COMP_PATH / "test.csv"
    )

    # Preprocess data
    (
        X,
        y,
        X_test,
        test_ids,
    ) = preprocess_data(
        train_df,
        test_df,
        TARGET,
    )

    # Split validation
    (
        X_train,
        X_valid,
        y_train,
        y_valid,
    ) = train_test_split(
        X,
        y,
        test_size=0.1,
        random_state=42,
    )

    # Convert train tensors
    X_train_tensor = torch.tensor(
        X_train,
        dtype=torch.float32,
    )

    y_train_tensor = torch.tensor(
        y_train,
        dtype=torch.float32,
    )

    # Convert validation tensors
    X_valid_tensor = torch.tensor(
        X_valid,
        dtype=torch.float32,
    )

    y_valid_tensor = torch.tensor(
        y_valid,
        dtype=torch.float32,
    )

    # Create train dataset
    train_dataset = TensorDataset(
        X_train_tensor,
        y_train_tensor,
    )

    # Create validation dataset
    valid_dataset = TensorDataset(
        X_valid_tensor,
        y_valid_tensor,
    )

    # Create train loader
    train_loader = DataLoader(
        train_dataset,
        batch_size=4096,
        shuffle=True,
    )

    # Create validation loader
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=4096,
        shuffle=False,
    )

    # Define device
    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    # Print device
    print(
        f"Using device: {device}"
    )

    # Define model
    model = ResidualRegressor(
        input_dim=X.shape[1],
        hidden_dim=96,
        dropout=0.2,
    )

    # Move model to device
    model.to(device)

    # Train model
    model = train_model(
        model=model,
        train_loader=train_loader,
        valid_loader=valid_loader,
        device=device,
        epochs=16,
        learning_rate=1e-3,
    )

    # Predict neural network
    nn_predictions = predict_model(
        model=model,
        X_test=X_test,
        device=device,
    )

    # Define prediction dictionary
    prediction_dict = {
        "sub1": {
            "predictions": pd.read_csv(
                BLEND_PATH
                / "submission.csv"
            )[TARGET].values,
            "weight": 2.9,
        },
        "sub2": {
            "predictions": pd.read_csv(
                BLEND_PATH
                / "submission (1).csv"
            )[TARGET].values,
            "weight": 0.1,
        },
        "nn": {
            "predictions": nn_predictions,
            "weight": 0.0000001,
        },
    }

    # Blend predictions
    blend_predictions(
        test_ids=test_ids,
        prediction_dict=prediction_dict,
        output_path="submission.csv",
        target_col=TARGET,
    )


# Call the main function
if __name__ == "__main__":
    main()

Using device: cpu
Epoch 01 | Train Loss: 0.21653 | Valid MAE: 0.15803
Epoch 02 | Train Loss: 0.15307 | Valid MAE: 0.13269
Epoch 03 | Train Loss: 0.13362 | Valid MAE: 0.12209
Epoch 04 | Train Loss: 0.12672 | Valid MAE: 0.12905
Epoch 05 | Train Loss: 0.12238 | Valid MAE: 0.11860
Epoch 06 | Train Loss: 0.12029 | Valid MAE: 0.11914
Epoch 07 | Train Loss: 0.11698 | Valid MAE: 0.11687
Epoch 08 | Train Loss: 0.11542 | Valid MAE: 0.11757
Epoch 09 | Train Loss: 0.11312 | Valid MAE: 0.11720
Epoch 10 | Train Loss: 0.11129 | Valid MAE: 0.11438
Epoch 11 | Train Loss: 0.11032 | Valid MAE: 0.11493
Epoch 12 | Train Loss: 0.10944 | Valid MAE: 0.11478
Epoch 13 | Train Loss: 0.10861 | Valid MAE: 0.11430
Epoch 14 | Train Loss: 0.10794 | Valid MAE: 0.12048
Epoch 15 | Train Loss: 0.10871 | Valid MAE: 0.11410
Epoch 16 | Train Loss: 0.10688 | Valid MAE: 0.11312
Blending sub1 | Weight: 2.9
Blending sub2 | Weight: 0.1
Blending nn | Weight: 1e-07
✅ Submission saved to submission.csv
